# Lekcija 08 - Višeagentski obrazac dizajna


## Postavljanje


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Zašto sustavi s više agenata?

Stvarni zadaci poput planiranja putovanja uključuju mnogo različitih vrsta stručnosti — logistiku, lokalno znanje, budžetiranje i još mnogo toga. Pojedinačni agent koji pokušava sve sam upravljati brzo postaje nezgrapan.

Sustavi s više agenata to rješavaju kroz **specijalizaciju**: svaki agent se fokusira na jedno područje stručnosti, dajući kvalitetnije rezultate nego generalist. Također poboljšavaju **skalabilnost** — možete dodati nove agente (npr. stručnjak za letove, kritičar restorana) bez prepisivanja postojećeg tijeka rada. Agenti se povezuju kroz strukturirani proces, prenoseći kontekst s jednog na drugog.


## Izrada specijaliziranih agenata


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Izgradnja sekvencijalnog tijeka rada

`WorkflowBuilder` vam omogućuje da spojite agente u usmjereni graf. Ovdje stvaramo jednostavan dvostepeni proces: **TravelPlanner** sastavlja plan puta, zatim ga **TravelConcierge** pregledava i unapređuje.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodavanje Više Agenata u Radni Tok

Jedna od najvećih prednosti multi-agentnog obrasca je koliko je lako proširiti ga. Ispod dodajemo agenta **BudgetReviewer** koji provjerava plan u odnosu na budžet putnika, označava stavke koje bi mogle premašiti troškove, i predlaže alternative za uštedu novca. Radni tok sada izvodi tri agenta u nizu:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Sažetak

U ovoj lekciji ste naučili kako:

1. **Kreirati specijalizirane agente** — svaki s usmjerenom ulogom (planiranje, portir, pregled proračuna).
2. **Povezati agente u sekvencijalni tijek rada** koristeći `WorkflowBuilder` i `add_edge`.
3. **Strimirati izlaz** iz višestrukog agentskog lanca, prateći koji agent govori.
4. **Proširiti tijek rada** dodavanjem novih agenata u lanac bez izmjene postojećih.

Dizajn uzorka višestrukih agenata održava svakog agenta jednostavnim, dok proizvodi bogatije, detaljnije pregledane rezultate nego što bi to mogao postići bilo koji pojedinačni agent samostalno.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
